In [ ]:
import pandas as pd
import numpy as np
import dask.dataframe as dd

In [ ]:
df = dd.read_parquet("../Dataset/CuratedQM9/new_qm9.parquet", engine="pyarrow")

In [ ]:
npy = np.load("../Dataset/qm9-or.npy", allow_pickle=True)

In [ ]:
npy_df = pd.DataFrame(npy.tolist())
npy_df = npy_df.rename(columns={"index": "molecule_id"})
npy_df["molecule_id"] = (
    npy_df["molecule_id"]
    .astype(str)
    .str.rsplit("_", n=1).str[-1]
    .str.extract(r"(\d+)", expand=False)
    .str.zfill(6)
)
npdd = dd.from_pandas(npy_df, npartitions=max(1, len(npy_df) // 10000))

In [ ]:
df['molecule_id'] = (
    df['molecule_id']
    .astype(str)
    .str.rsplit('_', n=1).str[-1]
    .str.extract(r'(\d+)', expand=False)
    .str.zfill(6)
)

In [ ]:
df_merge = df.assign(molecule_id=df["molecule_id"].astype("string"))
npdd_merge = npdd.assign(molecule_id=npdd["molecule_id"].astype("string"))
data = dd.merge(df_merge, npdd_merge, on="molecule_id", how="inner").compute()


In [ ]:
data = data.drop(['xyz', 'inchi'], axis=1)

In [ ]:
data = data.drop('source_file', axis=1)

In [ ]:
data.columns

In [ ]:
data.to_parquet("../Dataset/New_QM9/data.parquet", engine="pyarrow")